## Transform Orders Data - Explode Arrays
1. Access elements from the JSON object
1. Deduplicate Array Elements
1. Explode Arrays
1. Write the Transformed Data to Silver Schema

In [0]:
df_orders = spark.table('gizmobox.silver.py_orders_json')
display(df_orders)

json_value
"List(6973, List(List(Electronics, List(Sony, Blue), 8, Gaming Console, 499, 1)), 2025-01-05, 1, Completed, Bank Transfer, 499, 2025-01-05 10:13:59)"
"List(3532, List(List(Electronics, List(Canon, Gray), 4, Smartwatch, 299, 2), List(Electronics, List(Dell, Blue), 7, External Hard Drive, 129, 3)), 2025-01-19, 2, Cancelled, PayPal, 985, 2025-01-19 00:05:13)"
"List(3532, List(List(Electronics, List(Apple, White), 3, Wireless Headphones, 199, 3)), 2025-01-08, 3, Completed, Bank Transfer, 597, 2025-01-08 23:11:00)"
"List(3532, List(List(Electronics, List(Microsoft, Black), 2, Laptop, 999, 1)), 2025-01-05, 4, Cancelled, Bank Transfer, 999, 2025-01-05 05:49:26)"
"List(9179, List(List(Electronics, List(LG, White), 5, Tablet, 399, 1)), 2025-01-02, 8, Completed, PayPal, 399, 2025-01-02 13:11:15)"
"List(7207, List(List(Electronics, List(Bose, Blue), 3, Wireless Headphones, 199, 3)), 2025-01-01, 14, Cancelled, PayPal, 597, 2025-01-01 18:22:26)"
"List(4468, List(List(Electronics, List(Dell, Black), 10, Drone, 799, 1)), 2025-01-22, 18, Pending, Credit Card, 799, 2025-01-22 21:12:08)"
"List(4761, List(List(Electronics, List(HP, Silver), 3, Wireless Headphones, 199, 1)), 2025-01-01, 22, Pending, PayPal, 199, 2025-01-01 15:49:19)"
"List(7007, List(List(Electronics, List(Samsung, Silver), 1, Smartphone, 699, 1)), 2025-01-05, 26, Shipped, Bank Transfer, 699, 2025-01-05 19:01:25)"
"List(5953, List(List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1), List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1)), 2025-01-18, 29, Pending, Bank Transfer, 129, 2025-01-18 03:05:04)"


### 1. Access elements from the JSON object

In [0]:
from pyspark.sql import functions as F

In [0]:
df_orders_normalized = (
    df_orders
    .select (
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        "json_value.items"
    )
)
display(df_orders_normalized)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,items
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,"List(List(Electronics, List(Sony, Blue), 8, Gaming Console, 499, 1))"
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,"List(List(Electronics, List(Canon, Gray), 4, Smartwatch, 299, 2), List(Electronics, List(Dell, Blue), 7, External Hard Drive, 129, 3))"
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,"List(List(Electronics, List(Apple, White), 3, Wireless Headphones, 199, 3))"
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,"List(List(Electronics, List(Microsoft, Black), 2, Laptop, 999, 1))"
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,"List(List(Electronics, List(LG, White), 5, Tablet, 399, 1))"
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,"List(List(Electronics, List(Bose, Blue), 3, Wireless Headphones, 199, 3))"
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,"List(List(Electronics, List(Dell, Black), 10, Drone, 799, 1))"
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,"List(List(Electronics, List(HP, Silver), 3, Wireless Headphones, 199, 1))"
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,"List(List(Electronics, List(Samsung, Silver), 1, Smartphone, 699, 1))"
29,Pending,Bank Transfer,129,2025-01-18 03:05:04,5953,"List(List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1), List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1))"


### 2. Deduplicate Array Elements

In [0]:
df_orders_normalized = (
    df_orders
    .select (
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        F.array_distinct("json_value.items").alias("items")
    )
)
display(df_orders_normalized)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,items
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,"List(List(Electronics, List(Sony, Blue), 8, Gaming Console, 499, 1))"
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,"List(List(Electronics, List(Canon, Gray), 4, Smartwatch, 299, 2), List(Electronics, List(Dell, Blue), 7, External Hard Drive, 129, 3))"
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,"List(List(Electronics, List(Apple, White), 3, Wireless Headphones, 199, 3))"
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,"List(List(Electronics, List(Microsoft, Black), 2, Laptop, 999, 1))"
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,"List(List(Electronics, List(LG, White), 5, Tablet, 399, 1))"
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,"List(List(Electronics, List(Bose, Blue), 3, Wireless Headphones, 199, 3))"
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,"List(List(Electronics, List(Dell, Black), 10, Drone, 799, 1))"
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,"List(List(Electronics, List(HP, Silver), 3, Wireless Headphones, 199, 1))"
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,"List(List(Electronics, List(Samsung, Silver), 1, Smartphone, 699, 1))"
29,Pending,Bank Transfer,129,2025-01-18 03:05:04,5953,"List(List(Electronics, List(Bose, Gray), 7, External Hard Drive, 129, 1))"


### 3. Explode Arrays

In [0]:
df_orders_exploded = (
    df_orders_normalized
    .select (
        "order_id",
        "order_status",
        "payment_method",
        "total_amount",
        "transaction_timestamp",
        "customer_id",
        F.explode("items").alias("item")
    )
)

display(df_orders_exploded)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,"List(Electronics, List(Sony, Blue), 8, Gaming Console, 499, 1)"
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,"List(Electronics, List(Canon, Gray), 4, Smartwatch, 299, 2)"
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,"List(Electronics, List(Dell, Blue), 7, External Hard Drive, 129, 3)"
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,"List(Electronics, List(Apple, White), 3, Wireless Headphones, 199, 3)"
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,"List(Electronics, List(Microsoft, Black), 2, Laptop, 999, 1)"
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,"List(Electronics, List(LG, White), 5, Tablet, 399, 1)"
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,"List(Electronics, List(Bose, Blue), 3, Wireless Headphones, 199, 3)"
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,"List(Electronics, List(Dell, Black), 10, Drone, 799, 1)"
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,"List(Electronics, List(HP, Silver), 3, Wireless Headphones, 199, 1)"
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,"List(Electronics, List(Samsung, Silver), 1, Smartphone, 699, 1)"


In [0]:
df_order_items = (
    df_orders_exploded
    .select (
        "order_id",
        "order_status",
        "payment_method",
        "total_amount",
        "transaction_timestamp",
        "customer_id",
        "item.item_id",
        "item.name",
        "item.price",
        "item.quantity",
        "item.category",
        "item.details.brand",
        "item.details.color"
    )
)
display(df_order_items)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item_id,name,price,quantity,category,brand,color
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,8,Gaming Console,499,1,Electronics,Sony,Blue
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,4,Smartwatch,299,2,Electronics,Canon,Gray
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,7,External Hard Drive,129,3,Electronics,Dell,Blue
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,3,Wireless Headphones,199,3,Electronics,Apple,White
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,2,Laptop,999,1,Electronics,Microsoft,Black
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,5,Tablet,399,1,Electronics,LG,White
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,3,Wireless Headphones,199,3,Electronics,Bose,Blue
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,10,Drone,799,1,Electronics,Dell,Black
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,3,Wireless Headphones,199,1,Electronics,HP,Silver
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,1,Smartphone,699,1,Electronics,Samsung,Silver


### 4. Write the Transformed Data to Silver Schema

In [0]:
df_order_items.writeTo("gizmobox.silver.py_orders").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_orders;

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item_id,name,price,quantity,category,brand,color
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,8,Gaming Console,499,1,Electronics,Sony,Blue
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,4,Smartwatch,299,2,Electronics,Canon,Gray
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,7,External Hard Drive,129,3,Electronics,Dell,Blue
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,3,Wireless Headphones,199,3,Electronics,Apple,White
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,2,Laptop,999,1,Electronics,Microsoft,Black
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,5,Tablet,399,1,Electronics,LG,White
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,3,Wireless Headphones,199,3,Electronics,Bose,Blue
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,10,Drone,799,1,Electronics,Dell,Black
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,3,Wireless Headphones,199,1,Electronics,HP,Silver
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,1,Smartphone,699,1,Electronics,Samsung,Silver
